# Session 8 - pandas

pandas is the go-to library for data manipulation and analysis in Python. If NumPy is the engine, pandas is the dashboard, steering wheel, and GPS all in one.

Built on top of NumPy, pandas adds two powerful data structures:

- **Series** — a 1-D labeled array (like a single column in a spreadsheet)
- **DataFrame** — a 2-D labeled table (like a whole spreadsheet)

The labels (called *indexes*) are what make pandas so powerful. Instead of remembering that column 3 is "age," you just use `df["age"]`.

We'll work through the essential pandas operations using the Pokemon dataset as our running example.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Loading Data

The first step in any data project is loading data. pandas can read from CSV, Excel, JSON, SQL databases, HTML tables, and more. The most common is `read_csv()`:

In [ ]:
df = pd.read_csv("pokemon/pokemon.csv")
print(f"Shape: {df.shape}")  # (rows, columns)
print(f"Columns: {list(df.columns[:5])} ...")  # Show first 5 column names

### First Things First: `info()` and `describe()`

Before doing anything, always run these two methods. They tell you what you're working with:

In [ ]:
df.info()

From `info()` we can see:
- 801 rows, 41 columns
- Some columns have missing values (e.g., `type2` has only 417 non-null values)
- Most columns are `float64` or `int64`, but a few are `object` (strings)
- Memory usage: ~257 KB

Now let's get statistical summaries of the numeric columns:

In [ ]:
df.describe()

And a quick peek at the first few rows:

In [ ]:
df.head()

In [ ]:
# Test: Verify data loading
assert df.shape == (801, 41), f"Expected (801, 41), got {df.shape}"
assert "name" in df.columns
assert "type1" in df.columns
assert df["hp"].dtype == np.int64
assert df["attack"].dtype == np.int64
assert len(df) == 801
print("Data loading tests passed!")

## Series vs DataFrame

Understanding the difference is important:

- A **DataFrame** is a table (2D)
- A **Series** is a single column (1D)

When you select a single column from a DataFrame, you get a Series:

In [ ]:
# Selecting a column returns a Series
names = df["name"]
print(f"Type: {type(names)}")
print(f"Shape: {names.shape}")
print(f"First 5:\n{names.head()}")

In [ ]:
# Selecting multiple columns returns a DataFrame
subset = df[["name", "type1", "hp"]]
print(f"Type: {type(subset)}")
print(f"Shape: {subset.shape}")
subset.head()

## Selecting Rows: `.loc` and `.iloc`

There are two ways to select rows:

- `.iloc[]` — select by **integer position** (row number)
- `.loc[]` — select by **label** (index value)

Right now our index is just row numbers (0, 1, 2, ...), so they work the same:

In [ ]:
# Get row 10 by position
row = df.iloc[10]
print(f"Pokemon at row 10: {row['name']}")
print(f"Type: {row['type1']}")
print(f"HP: {row['hp']}")

But what if we set the Pokédex number as the index? Then `.loc` becomes much more intuitive:

In [ ]:
# Set pokedex_number as the index
df = df.set_index("pokedex_number")
print(f"Index name: {df.index.name}")
print(f"First 5 index values: {list(df.index[:5])}")

In [ ]:
# Now .loc uses the pokedex number directly
pikachu = df.loc[25]
print(f"Pokemon #25: {pikachu['name']}")
print(f"Type: {pikachu['type1']}")
print(f"Attack: {pikachu['attack']}")

In [ ]:
# Select multiple rows by label
df.loc[[1, 4, 7], ["name", "type1", "attack"]]

In [ ]:
# Test: Verify .loc and .iloc
# Reset for testing
test_df = pd.DataFrame({
    "name": ["A", "B", "C"],
    "value": [10, 20, 30]
})

assert test_df.iloc[0]["name"] == "A"
assert test_df.iloc[1]["value"] == 20

# With custom index
test_df = test_df.set_index("name")
assert test_df.loc["B"]["value"] == 20
assert test_df.loc["C"]["value"] == 30
print("loc/illoc tests passed!")

## Filtering with Boolean Indexing

This is how you subset data — the pandas equivalent of SQL's `WHERE` clause. Create a boolean condition and use it to filter rows:

In [ ]:
# All Pokemon lighter than 25 kg
light = df[df["weight_kg"] < 25]
print(f"Pokemon under 25kg: {len(light)} out of {len(df)}")
light[["name", "type1", "weight_kg"]].head(10)

In [ ]:
# Combine conditions with & (AND) and | (OR)
# Note: Each condition needs parentheses!
fire_and_heavy = df[(df["type1"] == "fire") & (df["weight_kg"] > 50)]
print(f"Fire types over 50kg: {len(fire_and_heavy)}")
fire_and_heavy[["name", "weight_kg", "attack"]]

### Using `.query()` for Readability

For complex filters, `.query()` is often cleaner:

In [ ]:
# Same filter, different syntax
light_query = df.query("weight_kg < 25")
assert len(light) == len(light_query), "query() should match boolean indexing"

# Complex query: legendary pokemon with speed over 100
fast_legendaries = df.query("is_legendary == 1 and speed > 100")
print(f"Fast legendaries (speed > 100): {len(fast_legendaries)}")
fast_legendaries[["name", "speed", "type1"]]

In [ ]:
# Test: Verify filtering
assert len(df[df["hp"] > 100]) > 0
assert len(df.query("hp > 100")) == len(df[df["hp"] > 100])

# Combined conditions
result = df[(df["attack"] > 100) & (df["defense"] > 100)]
assert all(result["attack"] > 100)
assert all(result["defense"] > 100)
print("Filtering tests passed!")

## Creating New Columns

Just like a dictionary, you can add new columns by assigning to them. Because of NumPy broadcasting underneath, vectorized operations are fast:

In [ ]:
# Z-score of base_total
df["base_total_zscore"] = (df["base_total"] - df["base_total"].mean()) / df["base_total"].std()

# Attack/Defense ratio
df["atk_def_ratio"] = df["attack"] / df["defense"]

# Size category
df["size_category"] = pd.cut(
    df["weight_kg"],
    bins=[0, 10, 50, 100, float("inf")],
    labels=["Tiny", "Small", "Medium", "Heavy"]
)

df[["name", "base_total", "base_total_zscore", "atk_def_ratio", "size_category"]].head(10)

In [ ]:
# Test: Verify column creation
assert "base_total_zscore" in df.columns
assert "atk_def_ratio" in df.columns
assert "size_category" in df.columns

# Z-score should have mean ~0
assert abs(df["base_total_zscore"].mean()) < 0.01

# Ratio should be positive
assert all(df["atk_def_ratio"] > 0)
print("Column creation tests passed!")

## Using `.apply()` and `.map()`

When vectorized operations aren't enough, `.apply()` lets you run custom functions on rows or columns. `.map()` is for transforming values in a Series:

In [ ]:
# Map: capitalize type names consistently
df["type1_clean"] = df["type1"].str.title()

# Apply: custom function on each row
def get_power_rating(row):
    """Calculate a simple power rating from base stats.
    
    Args:
        row: A DataFrame row with stat columns.
        
    Returns:
        A string rating.
    """
    total = row["base_total"]
    if total >= 600:
        return "Legendary"
    elif total >= 500:
        return "Strong"
    elif total >= 400:
        return "Average"
    else:
        return "Weak"


df["power_rating"] = df.apply(get_power_rating, axis=1)

# Show the distribution
print(df["power_rating"].value_counts())

In [ ]:
# Test: Verify apply and map
test_data = pd.DataFrame({"name": ["A", "B", "C"], "base_total": [300, 450, 650]})
test_data["power_rating"] = test_data.apply(get_power_rating, axis=1)
assert test_data.loc[0, "power_rating"] == "Weak"
assert test_data.loc[1, "power_rating"] == "Average"
assert test_data.loc[2, "power_rating"] == "Legendary"
print("Apply and map tests passed!")

## Handling Missing Data

Real data is messy. pandas makes it easy to find and deal with missing values:

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

The `type2` column has lots of missing values — that makes sense, since not all Pokemon have a second type. For `height_m` and `weight_kg`, the missing values are data gaps.

Common strategies:
- **Drop** rows with missing values: `df.dropna()`
- **Fill** with a value: `df.fillna(0)`
- **Fill** with a statistic: `df.fillna(df.median())`

In [ ]:
# Fill missing height/weight with the median
df["height_m"] = df["height_m"].fillna(df["height_m"].median())
df["weight_kg"] = df["weight_kg"].fillna(df["weight_kg"].median())

# Verify no more missing values in these columns
print(f"Missing height: {df['height_m'].isnull().sum()}")
print(f"Missing weight: {df['weight_kg'].isnull().sum()}")

In [ ]:
# Test: Verify missing data handling
test_series = pd.Series([1, 2, np.nan, 4, np.nan])
filled = test_series.fillna(test_series.median())
assert filled.isnull().sum() == 0
assert filled[2] == 2.0  # Median of [1, 2, 4] is 2

# dropna
dropped = test_series.dropna()
assert len(dropped) == 3
print("Missing data tests passed!")

## GroupBy and Aggregation

This is where pandas really shines. The split-apply-combine pattern lets you answer questions like "What's the average HP for each type?" in one line:

1. **Split** the data into groups
2. **Apply** an aggregation function to each group
3. **Combine** the results into a new DataFrame

In [ ]:
# Average stats by type
type_stats = df.groupby("type1")[["hp", "attack", "defense", "speed"]].mean().round(1)
type_stats

In [ ]:
# Multiple aggregations at once
type_summary = df.groupby("type1").agg(
    count=("name", "count"),
    avg_attack=("attack", "mean"),
    max_attack=("attack", "max"),
    avg_speed=("speed", "mean"),
).round(1)

# Sort by average attack
type_summary.sort_values("avg_attack", ascending=False)

In [ ]:
# Which type has the highest total stats on average?
df.groupby("type1")["base_total"].mean().sort_values(ascending=False).head(10)

In [ ]:
# Test: Verify groupby operations
test_df = pd.DataFrame({
    "group": ["A", "A", "B", "B"],
    "value": [10, 20, 30, 40]
})

# Mean by group
means = test_df.groupby("group")["value"].mean()
assert means["A"] == 15.0
assert means["B"] == 35.0

# Multiple aggregations
agg = test_df.groupby("group").agg(
    count=("value", "count"),
    total=("value", "sum")
)
assert agg.loc["A", "count"] == 2
assert agg.loc["A", "total"] == 30
print("GroupBy tests passed!")

## Sorting and Ranking

Quickly find the top entries:

In [ ]:
# Top 10 by attack
top_attack = df.nlargest(10, "attack")[["name", "type1", "attack", "base_total"]]
print("Top 10 by attack:")
top_attack

In [ ]:
# Rank all Pokemon by base_total
df["rank"] = df["base_total"].rank(ascending=False, method="min").astype(int)

# Show the top 5
df.nsmallest(5, "rank")[["name", "base_total", "rank"]]

In [ ]:
# Test: Verify sorting and ranking
test_df = pd.DataFrame({"name": ["A", "B", "C"], "score": [30, 10, 20]})

top = test_df.nlargest(1, "score")
assert top.iloc[0]["name"] == "A"

bottom = test_df.nsmallest(1, "score")
assert bottom.iloc[0]["name"] == "B"

# Rank
test_df["rank"] = test_df["score"].rank(ascending=False).astype(int)
assert test_df.loc[0, "rank"] == 1  # A has highest score
print("Sorting and ranking tests passed!")

## Value Counts and Crosstabs

Quick frequency analysis:

In [ ]:
# Distribution of primary types
type_counts = df["type1"].value_counts()
print("Type distribution:")
print(type_counts)

In [ ]:
# Crosstab: type vs legendary status
pd.crosstab(df["type1"], df["is_legendary"], rownames=["Type"], colnames=["Legendary"])

In [ ]:
# Test: Verify value_counts and crosstab
test_series = pd.Series(["a", "a", "b", "c", "a"])
counts = test_series.value_counts()
assert counts["a"] == 3
assert counts["b"] == 1

# Crosstab
ct = pd.crosstab(pd.Series(["X", "X", "Y"]), pd.Series([1, 0, 1]))
assert ct.loc["X", 0] == 1
assert ct.loc["Y", 1] == 1
print("Value counts and crosstab tests passed!")

## Combining DataFrames

In real projects you'll often need to merge data from multiple sources. pandas provides `concat()` to stack DataFrames and `merge()` to join them:

In [ ]:
# Create two small DataFrames
df1 = pd.DataFrame({"id": [1, 2, 3], "name": ["A", "B", "C"], "score": [90, 80, 70]})
df2 = pd.DataFrame({"id": [4, 5], "name": ["D", "E"], "score": [60, 50]})

# Stack them vertically (pd.concat replaces the deprecated .append)
combined = pd.concat([df1, df2], ignore_index=True)
print("Combined (concat):")
print(combined)

In [ ]:
# Join on a key column
grades = pd.DataFrame({"id": [1, 2, 3], "grade": ["A", "B", "C"]})
merged = df1.merge(grades, on="id", how="left")
print("Merged (merge on id):")
print(merged)

In [ ]:
# Test: Verify concat and merge
a = pd.DataFrame({"x": [1, 2]})
b = pd.DataFrame({"x": [3, 4]})
result = pd.concat([a, b], ignore_index=True)
assert len(result) == 4
assert list(result["x"]) == [1, 2, 3, 4]

# Merge
left = pd.DataFrame({"id": [1, 2], "val": ["a", "b"]})
right = pd.DataFrame({"id": [1, 2], "score": [100, 200]})
merged = left.merge(right, on="id")
assert len(merged) == 2
assert merged.loc[0, "val"] == "a"
assert merged.loc[0, "score"] == 100
print("Concat and merge tests passed!")

## String Methods

pandas gives you vectorized string operations through the `.str` accessor:

In [ ]:
# Clean up the abilities column: count how many abilities each Pokemon has
df["num_abilities"] = df["abilities"].str.count(",") + 1

# Find Pokemon whose names contain a specific pattern
df["name_length"] = df["name"].str.len()

# Extract the first ability
df["first_ability"] = df["abilities"].str.extract(r"'([^']+)'", expand=False)

df[["name", "abilities", "num_abilities", "first_ability", "name_length"]].head(10)

In [ ]:
# Test: Verify string methods
test_series = pd.Series(["hello", "WORLD", "Python"])
assert test_series.str.upper().iloc[0] == "HELLO"
assert test_series.str.lower().iloc[1] == "world"
assert test_series.str.contains("thon").iloc[2]
assert test_series.str.len().iloc[0] == 5
print("String method tests passed!")

## Visualization

pandas has built-in plotting powered by matplotlib. Just call `.plot()` on a Series or DataFrame:

In [ ]:
# Histogram of base_total
df["base_total"].plot(kind="hist", bins=30, edgecolor="black", figsize=(8, 4))
plt.title("Distribution of Base Total Stats")
plt.xlabel("Base Total")
plt.ylabel("Count")
plt.show()

In [ ]:
# Box plot: attack by type (top 5 types)
top_types = df["type1"].value_counts().head(5).index
subset = df[df["type1"].isin(top_types)]

subset.boxplot(column="attack", by="type1", figsize=(8, 5))
plt.title("Attack Distribution by Type")
plt.suptitle("")
plt.xlabel("Type")
plt.ylabel("Attack")
plt.show()

In [ ]:
# Scatter plot: attack vs defense, colored by legendary status
fig, ax = plt.subplots(figsize=(7, 5))

normal = df[df["is_legendary"] == 0]
legendary = df[df["is_legendary"] == 1]

ax.scatter(normal["attack"], normal["defense"], alpha=0.4, s=20, label="Normal", color="steelblue")
ax.scatter(legendary["attack"], legendary["defense"], alpha=0.7, s=40, label="Legendary", color="gold", edgecolors="black")

ax.set_xlabel("Attack")
ax.set_ylabel("Defense")
ax.set_title("Attack vs Defense")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Bar chart: average speed by generation
speed_by_gen = df.groupby("generation")["speed"].mean()
speed_by_gen.plot(kind="bar", figsize=(6, 4), color="teal", edgecolor="black")
plt.title("Average Speed by Generation")
plt.xlabel("Generation")
plt.ylabel("Average Speed")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Putting It All Together: A Mini Analysis

Let's answer a real question: **"Which Pokemon types have the best overall stats, and how do legendaries compare?"**

In [ ]:
# 1. Compute average base_total by type
type_avg = df.groupby("type1")["base_total"].mean().sort_values(ascending=False)

# 2. Compare legendary vs non-legendary averages
legendary_avg = df.groupby("is_legendary")["base_total"].mean()

print("Average Base Total by Type:")
print(type_avg.round(1))
print()
print(f"Non-legendary average: {legendary_avg[0]:.1f}")
print(f"Legendary average: {legendary_avg[1]:.1f}")
print(f"Legendaries are {legendary_avg[1]/legendary_avg[0]:.1f}x stronger on average")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Bar chart of type averages
type_avg.head(10).plot(kind="barh", ax=axes[0], color="coral")
axes[0].set_title("Top 10 Types by Avg Base Total")
axes[0].set_xlabel("Average Base Total")
axes[0].invert_yaxis()
axes[0].grid(axis="x", alpha=0.3)

# Right: Violin-like comparison (using box plot)
df.boxplot(column="base_total", by="is_legendary", ax=axes[1])
axes[1].set_title("Base Total: Normal vs Legendary")
axes[1].set_xticklabels(["Normal", "Legendary"])
axes[1].set_ylabel("Base Total")
plt.suptitle("")

plt.tight_layout()
plt.show()

## Working with NumPy Arrays

Remember that pandas is built on NumPy. You can always access the underlying array with `.values`:

In [ ]:
# Extract a column as a NumPy array
attack_array = df["attack"].values
print(f"Type: {type(attack_array)}")
print(f"Shape: {attack_array.shape}")
print(f"Mean (NumPy): {attack_array.mean():.1f}")
print(f"Mean (pandas): {df['attack'].mean():.1f}")

In [ ]:
# Convert entire DataFrame to NumPy array
stats = df[["hp", "attack", "defense", "sp_attack", "sp_defense", "speed"]].values
print(f"Stats array shape: {stats.shape}")
print(f"Mean of each stat: {stats.mean(axis=0).round(1)}")

## Summary

You've covered the essential pandas concepts:

- **Loading data** — `pd.read_csv()`, `df.info()`, `df.describe()`
- **Selecting data** — column selection, `.loc`, `.iloc`
- **Filtering** — boolean indexing, `.query()`
- **Creating columns** — vectorized operations, `.apply()`, `.map()`
- **Missing data** — `isnull()`, `fillna()`, `dropna()`
- **GroupBy** — `groupby()`, `agg()`, `value_counts()`
- **Sorting** — `nlargest()`, `nsmallest()`, `rank()`
- **Combining** — `pd.concat()`, `merge()`
- **String methods** — `.str.count()`, `.str.extract()`, `.str.len()`
- **Visualization** — `.plot()`, `.boxplot()`, matplotlib integration

### Quick Reference

| Task | pandas approach |
|------|----------------|
| Load CSV | `pd.read_csv("file.csv")` |
| First rows | `df.head()` |
| Summary stats | `df.describe()` |
| Select column | `df["column"]` |
| Select rows by position | `df.iloc[0:5]` |
| Select rows by label | `df.loc[label]` |
| Filter rows | `df[df["col"] > value]` |
| Create column | `df["new"] = df["a"] + df["b"]` |
| Group and aggregate | `df.groupby("col")["val"].mean()` |
| Multiple aggregations | `df.groupby("col").agg(...)` |
| Fill missing values | `df.fillna(value)` |
| Sort | `df.sort_values("col", ascending=False)` |
| Count values | `df["col"].value_counts()` |
| Concatenate | `pd.concat([df1, df2])` |
| Merge | `df1.merge(df2, on="key")` |
| Plot | `df["col"].plot(kind="hist")` |

You now have the foundation to explore real datasets. In the project notebooks, you'll put all of this together for Exploratory Data Analysis and machine learning.